# Day 037 — Exercise 1: drop_or_fill_nulls

**What you'll build:** `drop_or_fill_nulls(df, strategy) -> pd.DataFrame` — handle NaN values in numeric columns using one of four strategies: `'drop'` (remove rows), `'zero'` (replace with 0), `'mean'` (column mean), `'median'` (column median).

**Why it matters:** Missing values crash every downstream operation — groupby, ML models, arithmetic. Choosing the right fill strategy is the first engineering decision in any cleaning pipeline.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

# NULL_DF: rows 1 and 2 have nulls in different columns
NULL_DF = pd.DataFrame({
    'a':     [1.0, 2.0, None, 4.0],
    'b':     [10.0, None, 30.0, 40.0],
    'label': ['x', 'y', 'z', 'w'],
})

## Your Implementation

In [ ]:
def drop_or_fill_nulls(df: pd.DataFrame, strategy: str = 'mean') -> pd.DataFrame:
    """
    Handle NaN values in numeric columns.

    Strategies:
        'drop'   — remove any row that has at least one NaN; reset index
        'zero'   — replace NaN with 0 in all numeric columns
        'mean'   — replace NaN with the column mean
        'median' — replace NaN with the column median
    Non-numeric columns are not touched. Original df is not mutated.
    """
    result = df.copy()
    # TODO: if strategy == 'drop': return result.dropna().reset_index(drop=True)
    # TODO: num_cols = result.select_dtypes(include='number').columns
    # TODO: if strategy == 'zero':   result[num_cols] = result[num_cols].fillna(0)
    # TODO: elif strategy == 'mean': for col in num_cols: fill with col mean
    # TODO: elif strategy == 'median': for col in num_cols: fill with col median
    # TODO: else: raise ValueError(f'Unknown strategy ...')
    # TODO: return result
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined, returns a DataFrame
    try:
        assert 'drop_or_fill_nulls' in globals()
        result = drop_or_fill_nulls(NULL_DF, 'drop')
        assert isinstance(result, pd.DataFrame), \
            f'expected DataFrame, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: returns a DataFrame')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: strategy='drop' removes rows with any null
    try:
        result = drop_or_fill_nulls(NULL_DF, 'drop')
        assert len(result) == 2, f'expected 2 rows, got {len(result)}'
        assert result.isnull().sum().sum() == 0, 'nulls still present after drop'
        assert list(result.index) == [0, 1], f'index not reset: {list(result.index)}'
        passed += 1; print('\u2705 Check 2: strategy=drop removes null rows and resets index')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: strategy='zero' fills numeric nulls with 0
    try:
        result = drop_or_fill_nulls(NULL_DF, 'zero')
        assert result.shape == NULL_DF.shape, \
            f'shape changed: {result.shape}'
        assert result[['a', 'b']].isnull().sum().sum() == 0, \
            'numeric nulls not filled'
        assert result.at[1, 'b'] == 0.0, \
            f'b[1] should be 0.0, got {result.at[1, "b"]}'
        assert result.at[2, 'a'] == 0.0, \
            f'a[2] should be 0.0, got {result.at[2, "a"]}'
        passed += 1; print('\u2705 Check 3: strategy=zero fills nulls with 0')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: strategy='mean' fills with column mean
    try:
        result = drop_or_fill_nulls(NULL_DF, 'mean')
        assert result.shape == NULL_DF.shape
        assert result[['a', 'b']].isnull().sum().sum() == 0
        expected_a = (1 + 2 + 4) / 3
        actual_a   = float(result.at[2, 'a'])
        assert abs(actual_a - expected_a) < 0.001, \
            f'a[2] should be ~{expected_a:.3f}, got {actual_a:.3f}'
        passed += 1; print(f'\u2705 Check 4: strategy=mean fills a[2] with ~{expected_a:.3f}')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: original DataFrame is NOT mutated
    try:
        _ = drop_or_fill_nulls(NULL_DF, 'zero')
        assert NULL_DF.isnull().sum().sum() == 2, \
            'original DataFrame was mutated (use df.copy()!)'
        passed += 1; print('\u2705 Check 5: original DataFrame not mutated')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import pandas as pd

def drop_or_fill_nulls(df: pd.DataFrame, strategy: str = 'mean') -> pd.DataFrame:
    result   = df.copy()
    if strategy == 'drop':
        return result.dropna().reset_index(drop=True)
    num_cols = result.select_dtypes(include='number').columns
    if strategy == 'zero':
        result[num_cols] = result[num_cols].fillna(0)
    elif strategy == 'mean':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].mean())
    elif strategy == 'median':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].median())
    else:
        raise ValueError(
            f"Unknown strategy {strategy!r}. "
            "Use 'drop', 'zero', 'mean', or 'median'."
        )
    return result
```

</details>